# 00 — Download datasets (Slice 0 / Slice 1b prep)

Thin runner per `cultureQC_upgrade.md` §15: pinned commit, install, fetch, done.
Pins `slice-1b-compute-cache` at `816a098` — update `PINNED_SHA` below if that branch has moved
and you want a newer commit.

**What this notebook fetches** (all via reviewed `scripts/fetch_*.py` / existing
repo scripts — no unreviewed third-party download scripts run here):
- EVICAN `eval2019` (98 images, CC BY 4.0) — held-out real-data eval
- LIVECell images (CC BY-NC 4.0) — background source for `data/tiles/` synthesis
- `data/tiles/` regenerated from LIVECell (synthetic contamination/detachment/
  image-quality tiles, seed 42, 4000 tiles)
- AutoQC-Bench `test/` + `splits/` (MIT repo, BioStudies-hosted, ~53 MB)

**Not fetched here** — see `docs/DATASETS.md` for why:
- **C2C12** (Sci Data 2018, OSF-hosted): 48 sequences, ~49,919 images, tens–100+ GB.
  No automated fetcher exists yet (untested against the real OSF API) — fetch
  manually if/when Slice 2/3 actually needs it: `doi.org/10.17605/OSF.IO/YSAQ2`.
- **Adherent Cell Tracking Challenge**: blocked on organizer permission
  (`celltrackingchallenge.net` conditions of use require asking first).

Everything lands on Drive under `DRIVE_ROOT` so it survives Colab disconnects and
is reusable by `nb/02_compute_cache.ipynb` without re-fetching.

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/cultureqc'
import os
os.makedirs(DRIVE_ROOT, exist_ok=True)

In [ ]:
REPO_URL = 'https://github.com/n1tishc/cultureqc.git'
BRANCH = 'slice-1b-compute-cache'
PINNED_SHA = '816a098'

!rm -rf /content/cultureqc
!git clone --branch $BRANCH $REPO_URL /content/cultureqc
%cd /content/cultureqc
!git checkout $PINNED_SHA
!git rev-parse HEAD

In [ ]:
!pip install -q -r requirements.txt

import torch
print('torch', torch.__version__, '— CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), (
    'No GPU visible — Runtime > Change runtime type > GPU, then re-run this cell. '
    'This notebook only needs a GPU later (nb/02); dataset download itself is CPU-only, '
    'but catching a misconfigured runtime here saves a wasted round trip.'
)

## EVICAN — `eval2019` held-out split

In [ ]:
EVICAN_DIR = f'{DRIVE_ROOT}/data/sources/evican'
!python scripts/fetch_evican.py --out $EVICAN_DIR

## LIVECell — background source for `data/tiles/`

Public S3, no auth, CC BY-NC 4.0 (see `docs/DATASETS.md` — used only as sprite-
compositing background, never as a segmentation eval target).

In [ ]:
LIVECELL_DIR = f'{DRIVE_ROOT}/data/sources/livecell'
import os
images_dir = f'{LIVECELL_DIR}/images/livecell_train_val_images'
if not os.path.isdir(images_dir) or len(os.listdir(images_dir)) < 3700:
    os.makedirs(f'{LIVECELL_DIR}/images', exist_ok=True)
    !curl -s -o /content/livecell_images.zip \
        http://livecell-dataset.s3.eu-central-1.amazonaws.com/LIVECell_dataset_2021/images.zip
    !unzip -q -o /content/livecell_images.zip -d $LIVECELL_DIR/images
    !rm /content/livecell_images.zip
else:
    print('LIVECell already present:', len(os.listdir(images_dir)), 'files')
print(len(os.listdir(images_dir)), 'LIVECell train_val images')

## `data/tiles/` — regenerate synthetic tiles from LIVECell

Deterministic (`--seed 42`, the repo default) — verified on the Mac to reproduce
3/4 spot-checked `test-data/` fixtures bit-for-bit; the 1 mismatch is a benign,
understood library-version sensitivity in sprite placement, not a split-integrity
issue (see `docs/DATASETS.md`). DeepBacs sprites are fetched first — the source
`synth_contamination.py` composites onto LIVECell backgrounds.

In [ ]:
!python scripts/download_sources.py --out data/sources
!python scripts/extract_sprites.py --input data/sources/deepbacs --out data/sprites/bacteria

TILES_DIR = f'{DRIVE_ROOT}/data/tiles'
!python scripts/synth_contamination.py \
    --base-dir $images_dir \
    --sprite-dir data/sprites/bacteria \
    --out $TILES_DIR

In [ ]:
import pandas as pd
manifest = pd.read_csv(f'{TILES_DIR}/manifest.csv')
print(len(manifest), 'tiles')
print(manifest['class'].value_counts())

## AutoQC-Bench — `test/` + `splits/` only

In [ ]:
AUTOQC_DIR = f'{DRIVE_ROOT}/data/sources/autoqc_bench'
!python scripts/fetch_autoqc_bench.py --out $AUTOQC_DIR
# --include-train pulls the remaining ~2.8 GB (train/) — deliberately not done
# here; see docs/DATASETS.md for why that's a Slice 4 decision, not Slice 1b's.

## Summary

Run this last to confirm everything `nb/02_compute_cache.ipynb` expects is present
before switching notebooks.

In [ ]:
import os

def count(path, ext=None):
    if not os.path.isdir(path):
        return 0
    if ext:
        return sum(1 for f in os.listdir(path) if f.lower().endswith(ext))
    return len(os.listdir(path))

print('EVICAN eval2019 images:', count(f'{EVICAN_DIR}/eval2019_images', '.jpg'))
print('LIVECell train_val images:', count(images_dir, '.tif'))
print('data/tiles/ manifest rows:', len(pd.read_csv(f'{TILES_DIR}/manifest.csv')))
print('AutoQC-Bench test/ files:', count(f'{AUTOQC_DIR}/test/good_data/human'))
print()
print('All data under', DRIVE_ROOT)
print('Next: nb/02_compute_cache.ipynb')